In [ ]:
import os
import json
import csv
import numpy as np
from collections import defaultdict
from natsort import natsorted
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# Helper function (kept as provided)
def _process_sample_padded(sample_basename, layer_folders, npz_root, json_root, chunks_to_process):
    out = {}
    json_path = os.path.join(json_root, f"{sample_basename}.json")
    if not os.path.exists(json_path):
        return {}
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            tags_json = json.load(f)
    except Exception:
        return {}
    if not isinstance(tags_json, list) or len(tags_json) == 0:
        return {}
    token_tags = [tok.get('tag', 'OTHER') for tok in tags_json]

    for chunk in chunks_to_process:
        layer_arrays = []
        for layer in layer_folders:
            npz_path = os.path.join(npz_root, layer, "attention_progression", f"{sample_basename}.npz")
            if not os.path.exists(npz_path):
                continue
            try:
                data = np.load(npz_path, allow_pickle=True)
            except Exception:
                continue
            if chunk not in data:
                continue
            arr = np.array(data[chunk], dtype=object)
            arr = np.where(arr == None, np.nan, arr).astype(float)
            layer_arrays.append(arr)

        if not layer_arrays:
            continue

        max_len = max(a.shape[0] for a in layer_arrays)
        if max_len == 0:
            continue

        padded = np.full((len(layer_arrays), max_len), np.nan, dtype=float)
        for i, a in enumerate(layer_arrays):
            l = a.shape[0]
            padded[i, :l] = a

        with np.errstate(all='ignore'):
            per_token = np.nanmean(padded, axis=0)

        eff_seq_len = min(len(token_tags), per_token.shape[0])
        if eff_seq_len == 0:
            continue

        tags_slice = token_tags[:eff_seq_len]
        tag_to_inds = {}
        for idx, t in enumerate(tags_slice):
            tag_to_inds.setdefault(t, []).append(idx)

        sample_tag_means = {}
        for tag, inds in tag_to_inds.items():
            vals = per_token[inds]
            if np.all(np.isnan(vals)):
                continue
            sample_tag_means[tag] = float(np.nanmean(vals))

        if sample_tag_means:
            out[chunk] = sample_tag_means

    return out

def extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=False):
    """
    Computes POS-tag averaged attention stats and saves them to a CSV file.
    
    Columns in output CSV:
    [chunk, tag, mean, std, median, count, baseline_subtracted_mean (optional)]
    """

    # 1. Load Baseline (if any)
    baseline = {}
    if baseline_json_path is not None:
        if not os.path.exists(baseline_json_path):
            raise FileNotFoundError(f"Baseline JSON not found: {baseline_json_path}")
        with open(baseline_json_path, 'r') as f:
            baseline = json.load(f)
        baseline = {k: float(v) for k, v in baseline.items()}

    # 2. Discover layers
    if layer_folders is None:
        cand = [d for d in os.listdir(npz_root) if os.path.isdir(os.path.join(npz_root, d))]
        layer_folders = natsorted(cand)
    else:
        layer_folders = list(layer_folders)
    if not layer_folders:
        raise ValueError("No layer folders found in npz_root.")

    # 3. Discover samples
    if sample_list is None:
        sample_list = []
        for layer in layer_folders:
            att_dir = os.path.join(npz_root, layer, "attention_progression")
            if os.path.isdir(att_dir):
                files = [f for f in os.listdir(att_dir) if f.endswith('.npz')]
                sample_list = natsorted([os.path.splitext(f)[0] for f in files])
                break
    if max_samples is not None:
        sample_list = sample_list[:max_samples]
    if not sample_list:
        raise ValueError("No samples found.")

    # 4. Configure Workers
    if max_workers is None:
        try:
            import multiprocessing
            max_workers = min(8, multiprocessing.cpu_count() or 4)
        except Exception:
            max_workers = 4

    worker = partial(_process_sample_padded,
                     layer_folders=layer_folders,
                     npz_root=npz_root,
                     json_root=json_root,
                     chunks_to_process=chunks_to_process)

    collector = {chunk: defaultdict(list) for chunk in chunks_to_process}
    
    if verbose:
        print(f"Processing {len(sample_list)} samples with {max_workers} workers...")
        print(f"Target file: {output_csv_path}")

    # 5. Execute Parallel Processing
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(worker, sb): sb for sb in sample_list}
        for fut in as_completed(futures):
            sb = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                if verbose:
                    print(f"[error] sample {sb} -> {e}")
                continue
            if not res:
                continue
            for chunk, tagmap in res.items():
                for tag, val in tagmap.items():
                    if np.isnan(val):
                        continue
                    collector[chunk][tag].append(float(val))

    # 6. Aggregate Stats and Write to CSV
    # Ensure output directory exists
    output_dir = os.path.dirname(output_csv_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    rows_written = 0
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        
        # Define Header
        header = ['chunk', 'tag', 'mean', 'std', 'median', 'count']
        if subtract_baseline:
            header.append('mean_baseline_subtracted')
        
        writer.writerow(header)

        for chunk in chunks_to_process:
            # Sort tags for consistent output, though not strictly necessary for parsing
            # available_tags = sorted(collector[chunk].keys())
            
            available_tags = ["SOG", "FRUIT_INTRO", "FRUIT_CONCEPT", "FIRST_HANDOFF","OTHER_HANDOFF","MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
            # available_tags = ["SOG","MATH_INTRO", "MATH_ANSWER","FIRST_HANDOFF", "OTHER_HANDOFF", "FRUIT_INTRO","FRUIT_CONCEPT", "POSTAMBLE"]
            # available_tags=["SOG","FRUIT_INTRO","FRUIT_CONCEPT","FIRST_HANDOFF", "OTHER_HANDOFF","SPORT_INTRO", "SPORT_CONCEPT", "POSTAMBLE"]
            # available_tags = ["SOG", "SPORT_INTRO", "SPORT_CONCEPT", "FIRST_HANDOFF", "OTHER_HANDOFF", "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
            # available_tags = ["FRUIT_CONCEPT"]
            for tag in available_tags:
                # Apply tag filtering if requested
                if tags_to_include is not None and tag not in tags_to_include:
                    continue

                vals = collector[chunk][tag]
                arr = np.array(vals, dtype=float)
                
                if arr.size == 0:
                    continue

                mean_val = float(np.nanmean(arr))
                std_val = float(np.nanstd(arr)) if arr.size > 1 else 0.0
                median_val = float(np.nanmedian(arr))
                count_val = int(np.count_nonzero(~np.isnan(arr)))

                row = [chunk, tag, mean_val, std_val, median_val, count_val]

                if subtract_baseline:
                    if tag in baseline:
                        adjusted_mean = mean_val - baseline[tag]
                    else:
                        adjusted_mean = mean_val # Or np.nan if you prefer to indicate missing baseline
                    row.append(adjusted_mean)
                
                writer.writerow(row)
                rows_written += 1

    if verbose:
        print(f"Done. Wrote {rows_written} rows to {output_csv_path}")
    
    return collector # returning raw collector in case you want to inspect in memory

In [ ]:
npz_root = '<RUN_ROOT>/one_token_at_a_time/Rebuttal/Gemma3/fruit_math'
json_root = '<RUN_ROOT>/scaling_experiment_with_gemini/tagged_outputs_final/gemma3/image_blocking/fruit_concept/fruit_math/Gemma3'


# chunks_to_process=[
#         "currently_generating_token__attends_to__sport_text",
#         "currently_generating_token__attends_to__math_text",
#         "currently_generating_token__attends_to__instruction",
#         "currently_generating_token__attends_to__previously_generating_tokens",

#     ]

chunks_to_process=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__previously_generating_tokens",

    ]

# tags_to_include=["SOG", "SPORT_INTRO", "SPORT_CONCEPT", "FIRST_HANDOFF", "OTHER_HANDOFF", "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
tags_to_include=["SOG", "FRUIT_INTRO", "FRUIT_CONCEPT", "FIRST_HANDOFF", "OTHER_HANDOFF", "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
output_csv_path = 'bar_plot_csv_data/vanilla/fruit_math/gemma3_4b_it.csv'



extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=False)

# Layer-Grouped CSVs

In [ ]:
import os
import json
import csv
import numpy as np
from collections import defaultdict
from natsort import natsorted
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial


def _split_layers_into_groups(layer_folders):
    """Splits a sorted list of layer folders into early, mid, and late thirds."""
    n = len(layer_folders)
    if n == 0:
        return {}
    third = n // 3
    return {
        "early": layer_folders[:third],
        "mid":   layer_folders[third : 2 * third],
        "late":  layer_folders[2 * third :],
    }


def _process_sample_padded(sample_basename, layer_groups, npz_root, json_root, chunks_to_process):
    """
    Returns:
        dict[chunk][group_name][tag] -> float (per-sample mean attention value)
    """
    out = {}

    json_path = os.path.join(json_root, f"{sample_basename}.json")
    if not os.path.exists(json_path):
        return {}
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            tags_json = json.load(f)
    except Exception:
        return {}
    if not isinstance(tags_json, list) or len(tags_json) == 0:
        return {}
    token_tags = [tok.get('tag', 'OTHER') for tok in tags_json]

    for chunk in chunks_to_process:
        chunk_data = {}

        for group_name, layer_folders in layer_groups.items():
            layer_arrays = []
            for layer in layer_folders:
                npz_path = os.path.join(npz_root, layer, "attention_progression", f"{sample_basename}.npz")
                if not os.path.exists(npz_path):
                    continue
                try:
                    data = np.load(npz_path, allow_pickle=True)
                except Exception:
                    continue
                if chunk not in data:
                    continue
                arr = np.array(data[chunk], dtype=object)
                arr = np.where(arr == None, np.nan, arr).astype(float)
                layer_arrays.append(arr)

            if not layer_arrays:
                continue

            max_len = max(a.shape[0] for a in layer_arrays)
            if max_len == 0:
                continue

            padded = np.full((len(layer_arrays), max_len), np.nan, dtype=float)
            for i, a in enumerate(layer_arrays):
                padded[i, : a.shape[0]] = a

            with np.errstate(all='ignore'):
                per_token = np.nanmean(padded, axis=0)  # shape: (max_len,)

            eff_seq_len = min(len(token_tags), per_token.shape[0])
            if eff_seq_len == 0:
                continue

            tags_slice = token_tags[:eff_seq_len]
            tag_to_inds = {}
            for idx, t in enumerate(tags_slice):
                tag_to_inds.setdefault(t, []).append(idx)

            group_tag_means = {}
            for tag, inds in tag_to_inds.items():
                vals = per_token[inds]
                if np.all(np.isnan(vals)):
                    continue
                group_tag_means[tag] = float(np.nanmean(vals))

            if group_tag_means:
                chunk_data[group_name] = group_tag_means

        if chunk_data:
            out[chunk] = chunk_data

    return out


def extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=False):
    """
    Computes POS-tag averaged attention stats broken down by layer group
    (early / mid / late thirds) and saves them to a CSV file.

    Columns in output CSV:
        chunk, layer_group, tag, mean, std, median, count
        [, mean_baseline_subtracted]  <- only when subtract_baseline=True
    """

    # 1. Load baseline (if any)
    baseline = {}
    if baseline_json_path is not None:
        if not os.path.exists(baseline_json_path):
            raise FileNotFoundError(f"Baseline JSON not found: {baseline_json_path}")
        with open(baseline_json_path, 'r') as f:
            baseline = json.load(f)
        baseline = {k: float(v) for k, v in baseline.items()}

    # 2. Discover layers and split into groups
    if layer_folders is None:
        cand = [d for d in os.listdir(npz_root) if os.path.isdir(os.path.join(npz_root, d))]
        layer_folders = natsorted(cand)
    else:
        layer_folders = list(layer_folders)
    if not layer_folders:
        raise ValueError("No layer folders found in npz_root.")

    layer_groups = _split_layers_into_groups(layer_folders)

    if verbose:
        for g, layers in layer_groups.items():
            print(f"  [{g}] {len(layers)} layers: {layers[0]} … {layers[-1]}")

    # 3. Discover samples
    if sample_list is None:
        sample_list = []
        for layer in layer_folders:
            att_dir = os.path.join(npz_root, layer, "attention_progression")
            if os.path.isdir(att_dir):
                files = [f for f in os.listdir(att_dir) if f.endswith('.npz')]
                sample_list = natsorted([os.path.splitext(f)[0] for f in files])
                break
    if max_samples is not None:
        sample_list = sample_list[:max_samples]
    if not sample_list:
        raise ValueError("No samples found.")

    # 4. Configure workers
    if max_workers is None:
        try:
            import multiprocessing
            max_workers = min(8, multiprocessing.cpu_count() or 4)
        except Exception:
            max_workers = 4

    worker = partial(_process_sample_padded,
                     layer_groups=layer_groups,
                     npz_root=npz_root,
                     json_root=json_root,
                     chunks_to_process=chunks_to_process)

    # collector[chunk][group_name][tag] -> list of per-sample means
    GROUP_NAMES = ["early", "mid", "late"]
    collector = {
        chunk: {g: defaultdict(list) for g in GROUP_NAMES}
        for chunk in chunks_to_process
    }

    if verbose:
        print(f"Processing {len(sample_list)} samples with {max_workers} workers...")
        print(f"Target file: {output_csv_path}")

    # 5. Execute parallel processing
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(worker, sb): sb for sb in sample_list}
        for fut in as_completed(futures):
            sb = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                if verbose:
                    print(f"[error] sample {sb} -> {e}")
                continue
            if not res:
                continue
            for chunk, group_data in res.items():
                for group_name, tagmap in group_data.items():
                    for tag, val in tagmap.items():
                        if np.isnan(val):
                            continue
                        collector[chunk][group_name][tag].append(float(val))

    # 6. Aggregate stats and write CSV
    output_dir = os.path.dirname(output_csv_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    available_tags = [
        "SOG", "FRUIT_INTRO", "FRUIT_CONCEPT",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE",
    ]

    rows_written = 0
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        header = ['chunk', 'layer_group', 'tag', 'mean', 'std', 'median', 'count']
        if subtract_baseline:
            header.append('mean_baseline_subtracted')
        writer.writerow(header)

        for chunk in chunks_to_process:
            for group_name in GROUP_NAMES:
                for tag in available_tags:
                    if tags_to_include is not None and tag not in tags_to_include:
                        continue

                    vals = collector[chunk][group_name][tag]
                    arr = np.array(vals, dtype=float)
                    if arr.size == 0:
                        continue

                    mean_val   = float(np.nanmean(arr))
                    std_val    = float(np.nanstd(arr)) if arr.size > 1 else 0.0
                    median_val = float(np.nanmedian(arr))
                    count_val  = int(np.count_nonzero(~np.isnan(arr)))

                    row = [chunk, group_name, tag, mean_val, std_val, median_val, count_val]

                    if subtract_baseline:
                        adjusted = mean_val - baseline.get(tag, 0.0)
                        row.append(adjusted)

                    writer.writerow(row)
                    rows_written += 1

    if verbose:
        print(f"Done. Wrote {rows_written} rows to {output_csv_path}")

    return collector

In [ ]:
npz_root = '<RUN_ROOT>/<RUN_ROOT>/fruit_math/qwenv25vl3B'
json_root = 'tagged_outputs_final/vanilla/qwenv25vl3B/fruit_math'


chunks_to_process=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__previously_generating_tokens",

    ]

# tags_to_include=["SOG", "SPORT_INTRO", "SPORT_CONCEPT", "FIRST_HANDOFF", "OTHER_HANDOFF", "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
output_csv_path = 'data_plotting/bar_plot_csv_data_layerGrouped/vanilla/fruit_math/q25vl3B.csv'



extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=True)